
# 00 — Quickstart: SDGFT-ML-Toolkit

This notebook demonstrates the core workflow in under 5 minutes:

1. **Load** the GNN ensemble predictor
2. **Predict** all 37 observables at the axiom point
3. **Compare** predictions to 22 experimental measurements
4. **Query** the Oracle Database for best-fit parameter regions

**Prerequisites:** `pip install -e .` from the project root.

> **Oracle Database DOI:** [10.5281/zenodo.18863347](https://doi.org/10.5281/zenodo.18863347)


In [ ]:
# Install in Colab / fresh environment (skip if already installed)
# !pip install -e /path/to/sdgft-ml-toolkit

## 1. Load the GNN Ensemble Predictor

The `SDGFTPredictor` loads 5 trained GATv2 ensemble members (~1.3M params each)
and provides a simple prediction interface.

In [ ]:
from sdgft_ml.inference import SDGFTPredictor

predictor = SDGFTPredictor()   # auto-detects GPU, loads all 5 members
print(predictor)
print(f"\nModel info: {predictor.info}")

## 2. Predict at the Axiom Point

The SDGFT axiom fixes Δ = 5/24 ≈ 0.2083, δ_g = 1/24 ≈ 0.04167, φ = golden ratio.
One call returns all 37 observables.

In [ ]:
# Simple prediction — ensemble mean
result = predictor.predict()   # defaults to axiom point

print("Key predictions at the axiom point:")
print(f"  Higgs mass:          {result['higgs_mass']:.2f} GeV")
print(f"  sin²θ_W:            {result['sin2_theta_w']:.5f}")
print(f"  α_s(M_Z):           {result['alpha_s']:.4f}")
print(f"  n_s:                 {result['n_s']:.4f}")
print(f"  Ω_m:                {result['omega_m']:.4f}")
print(f"  Ω_Λ:                {result['omega_de']:.4f}")
print(f"  N_gen:              {result['n_generations']:.3f}")

In [ ]:
# Prediction with ensemble uncertainty (mean ± std)
result_unc = predictor.predict_with_uncertainty()

for name in ["higgs_mass", "sin2_theta_w", "n_s", "omega_m", "r_tensor"]:
    r = result_unc[name]
    print(f"  {name:<22s}  {r['mean']:>12.6g}  ± {r['std']:>10.6g}")

## 3. Compare to Experimental Data

The validation module compares theory predictions against 22 precision
measurements from PDG 2024, Planck 2018, NuFIT 5.3, and BICEP/Keck.

In [ ]:
from sdgft_ml.validation import validate_at_axiom, scorecard, chi_squared

# Run validation at the axiom point
results = validate_at_axiom()

# Print the full scorecard
scorecard(results)

In [ ]:
# χ² summary
chi2 = chi_squared(results)
print(f"\nχ² = {chi2['chi2']:.2f}  (N_dof = {chi2['ndof']}, χ²/dof = {chi2['chi2_per_dof']:.2f})")
print(f"p-value = {chi2['p_value']:.4f}")

print("\nPer-category breakdown:")
for cat, info in chi2['per_category'].items():
    print(f"  {cat:<14s}  χ²={info['chi2']:.2f}  dof={info['ndof']}  χ²/dof={info['chi2_per_dof']:.2f}")

## 4. Query the Oracle Database

The Oracle DB contains 61.7M+ pre-computed parameter points with χ² scores.
It enables instant exploration of the parameter landscape without re-running the GNN.

In [ ]:
from sdgft_ml.inference import OracleDB

db = OracleDB()   # lazy-loads on first access
print(db.summary())

In [ ]:
# Top 10 best-fit parameter points
top10 = db.best_fit(10)
top10[["delta", "delta_g", "total_chi2", "chi2_per_dof", "higgs_mass", "n_s", "omega_m"]]

In [ ]:
# Quick heatmap of the χ² landscape
import matplotlib.pyplot as plt
import numpy as np

grid, d_edges, dg_edges = db.chi2_heatmap(bins=200)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(
    grid.T, origin="lower", aspect="auto",
    extent=[d_edges[0], d_edges[-1], dg_edges[0], dg_edges[-1]],
    cmap="viridis_r", vmin=0, vmax=50,
)
ax.set_xlabel("Δ (Fibonacci-lattice conflict)")
ax.set_ylabel("δ_g (lattice tension)")
ax.set_title("SDGFT Oracle: Minimum χ² per bin")
ax.axvline(5/24, color='r', ls='--', alpha=0.7, label=f'Axiom Δ=5/24')
ax.axhline(1/24, color='r', ls='--', alpha=0.7, label=f'Axiom δ_g=1/24')
ax.legend()
plt.colorbar(im, label="min(χ²)")
plt.tight_layout()
plt.show()

---

## Next Steps

- **[01 Oracle Queries](01_oracle_queries.ipynb)** — Advanced filtering, DuckDB integration, export
- **[02 Parameter Landscape](02_parameter_landscape.ipynb)** — Sensitivity maps, correlation analysis
- **[03 Experimental Validation](03_experimental_validation.ipynb)** — Full 22-observable scorecard, pull distributions
- **[04 Predictions & Frontier](04_predictions_frontier.ipynb)** — W-boson mass, muon g-2, gravitational waves
- **[05 Inverse Problem](05_inverse_problem.ipynb)** — CVAE parameter recovery from observations